<a href="https://colab.research.google.com/github/pskshyam/Full-Stack-GenAI-Bootcamp-1.0/blob/main/Class-30-Data-Parsing-for-RAG/data_parsing_part2_pymupdf_pdfplumber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A complex PDF may contain normal text, scanned pages, tables and images. One parser may not extract everything correctly. Therefore, we use PyMuPDF for text and images, Tesseract for OCR, pdfplumber for tables, and finally convert the extracted content into LangChain Documents so it can be used in a RAG pipeline.

In [4]:
! pip install -q pymupdf pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 92.4 MB/s eta 0:00:00


In [5]:
# ============================================================
# Complex PDF Parser for RAG
# Text + Tables + Images + OCR + LangChain Documents
# ============================================================

# Install dependencies

import os
import json
import fitz  # PyMuPDF
import pdfplumber
import pandas as pd
from PIL import Image
from pathlib import Path
from langchain_core.documents import Document

In [6]:
# Install the Tesseract engine and Python library

!apt-get update
!apt-get install -y tesseract-ocr
!pip install pytesseract

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,057 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,548 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,729 k

In [7]:
import os
import pytesseract
from PIL import Image

TESSERACT_PATH = r"/usr/bin/tesseract"

if not os.path.exists(TESSERACT_PATH):
    raise FileNotFoundError(
        f"Tesseract executable not found at: {TESSERACT_PATH}"
    )

pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

print("Tesseract version:", pytesseract.get_tesseract_version())

Tesseract version: 5.3.4


In [8]:
# ============================================================
# 1. File Path
# ============================================================

PDF_PATH = "complex_rag_parsing_sample_with_sunny_image.pdf"

OUTPUT_DIR = Path("parsed_complex_pdf_output")

IMAGE_DIR = OUTPUT_DIR / "extracted_images"
PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("PDF exists:", os.path.exists(PDF_PATH))


PDF exists: True


In [9]:
# ============================================================
# 2. Helper: Safe OCR
# ============================================================

def run_ocr_on_image(image_path):
    """
    Runs OCR on image using pytesseract.
    If tesseract is not installed in system, it will return empty text.
    """
    try:
        import pytesseract
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        return text.strip()
    except Exception as e:
        return f"[OCR_SKIPPED_OR_FAILED: {str(e)}]"

In [10]:
run_ocr_on_image("page_002_image_1.png")

'Contract Processing Ownership Map\n\nClient Admin Audit Team\n\nVector DB Document Store'

In [11]:
# ============================================================
# 3. Extract Text + Images using PyMuPDF
# ============================================================

def extract_text_and_images(pdf_path):
    doc = fitz.open(pdf_path)

    page_records = []
    image_records = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_number = page_index + 1

        # Extract normal selectable text
        text = page.get_text("text")

        # Extract page metadata-like info
        page_info = {
            "page_number": page_number,
            "text": text.strip(),
            "image_count": len(page.get_images(full=True)),
            "width": page.rect.width,
            "height": page.rect.height,
        }

        page_records.append(page_info)

        # Render full page as image for OCR
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        page_image_path = PAGE_IMAGE_DIR / f"page_{page_number:03d}.png"
        pix.save(str(page_image_path))

        # OCR full page image
        ocr_text = run_ocr_on_image(page_image_path)
        page_info["ocr_text"] = ocr_text
        page_info["page_image_path"] = str(page_image_path)

        # Extract embedded images
        images = page.get_images(full=True)

        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = doc.extract_image(xref)

            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_path = IMAGE_DIR / f"page_{page_number:03d}_image_{img_index + 1}.{image_ext}"

            with open(image_path, "wb") as f:
                f.write(image_bytes)

            image_ocr_text = run_ocr_on_image(image_path)

            image_records.append({
                "page_number": page_number,
                "image_index": img_index + 1,
                "image_path": str(image_path),
                "image_ext": image_ext,
                "image_ocr_text": image_ocr_text
            })

    doc.close()

    return page_records, image_records


In [12]:
page_records, image_records = extract_text_and_images(PDF_PATH)

print("Total pages parsed:", len(page_records))
print("Total images extracted:", len(image_records))

Total pages parsed: 24
Total images extracted: 13


In [14]:
page_records[1]

{'page_number': 2,
 'text': 'Complex RAG Parsing Sample - synthetic document\nPage 2\n1. Business Scenario\nThe platform team is building a document intelligence layer for contract operations. Legal users need answers such as: Which\ncontracts renew automatically? Which agreements have a 60-day termination notice? Which documents mention data\nresidency in India? These questions require retrieval across multiple document types and sometimes across multiple related\ndocuments of a single client.\nSupport teams also want answers from SOPs, markdown runbooks, HTML exports, chat transcripts, and CSV logs. A single\nparser is not enough because the corpus contains born-digital PDFs, scanned forms, HTML, markdown, and spreadsheet-like\ntables.\nClient\nMain document family\nRisk\nPrimary users\nTypical question\nArka Finance\nMSA, audit addendum, DPA\nHigh\nLegal and compliance\nWhat audit rights exist in the DPA?\nBlueLeaf Retail\nMSA, pricing, support SLA\nMedium\nProcurement\nWhich clause

In [15]:
# ============================================================
# 4. Extract Tables using pdfplumber
# ============================================================

def extract_tables(pdf_path):
    table_records = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages):
            page_number = page_index + 1

            try:
                tables = page.extract_tables()
            except Exception as e:
                tables = []
                print(f"Table extraction failed on page {page_number}: {e}")

            for table_index, table in enumerate(tables):
                if not table:
                    continue

                # Clean table rows
                cleaned_table = []
                for row in table:
                    cleaned_row = [
                        cell.strip() if isinstance(cell, str) else cell
                        for cell in row
                    ]
                    cleaned_table.append(cleaned_row)

                # Convert to DataFrame safely
                try:
                    df = pd.DataFrame(cleaned_table[1:], columns=cleaned_table[0])
                except Exception:
                    df = pd.DataFrame(cleaned_table)

                table_records.append({
                    "page_number": page_number,
                    "table_index": table_index + 1,
                    "raw_table": cleaned_table,
                    "markdown": df.to_markdown(index=False),
                    "csv": df.to_csv(index=False)
                })

    return table_records

In [17]:
table_records[0]

{'page_number': 1,
 'table_index': 1,
 'raw_table': [['Section', 'Parsing challenge', 'Why it matters for RAG'],
  ['Contracts',
   'Dense legal text, clause numbers, cross\nreferences',
   'Need section-aware chunks and citations'],
  ['Tables',
   'Merged headers, numeric columns,\nfootnotes',
   'Need row/column preservation'],
  ['Images',
   'Architecture diagram, heatmap, scanned\nform',
   'Need OCR or multimodal extraction'],
  ['Multi-tenant metadata',
   'client_id, document_id,\ncontract_group_id',
   'Need access-safe retrieval']],
 'markdown': '| Section               | Parsing challenge                       | Why it matters for RAG                  |\n|:----------------------|:----------------------------------------|:----------------------------------------|\n| Contracts             | Dense legal text, clause numbers, cross | Need section-aware chunks and citations |\n|                       | references                              |                                    

In [16]:
table_records = extract_tables(PDF_PATH)

print("Total tables extracted:", len(table_records))

for table in table_records[:3]:
    print("\nPage:", table["page_number"], "Table:", table["table_index"])
    print(table["markdown"][:1000])

Total tables extracted: 16

Page: 1 Table: 1
| Section               | Parsing challenge                       | Why it matters for RAG                  |
|:----------------------|:----------------------------------------|:----------------------------------------|
| Contracts             | Dense legal text, clause numbers, cross | Need section-aware chunks and citations |
|                       | references                              |                                         |
| Tables                | Merged headers, numeric columns,        | Need row/column preservation            |
|                       | footnotes                               |                                         |
| Images                | Architecture diagram, heatmap, scanned  | Need OCR or multimodal extraction       |
|                       | form                                    |                                         |
| Multi-tenant metadata | client_id, document_id,                 | Need ac

In [18]:
# ============================================================
# 5. Create LangChain Documents
# ============================================================

langchain_docs = []

# 5.1 Page text documents
for page in page_records:
    page_number = page["page_number"]

    combined_text = f"""
PAGE {page_number}

SELECTABLE TEXT:
{page["text"]}

OCR TEXT:
{page["ocr_text"]}
""".strip()

    doc = Document(
        page_content=combined_text,
        metadata={
            "source": PDF_PATH,
            "page_number": page_number,
            "content_type": "page_text_plus_ocr",
            "image_count": page["image_count"],
            "page_image_path": page["page_image_path"],
        }
    )

    langchain_docs.append(doc)


# 5.2 Table documents
for table in table_records:
    page_number = table["page_number"]

    table_text = f"""
TABLE FOUND ON PAGE {page_number}
TABLE INDEX: {table["table_index"]}

TABLE MARKDOWN:
{table["markdown"]}
""".strip()

    doc = Document(
        page_content=table_text,
        metadata={
            "source": PDF_PATH,
            "page_number": page_number,
            "content_type": "table",
            "table_index": table["table_index"],
        }
    )

    langchain_docs.append(doc)


In [19]:
# 5.3 Image OCR documents
for image in image_records:
    page_number = image["page_number"]

    image_text = f"""
IMAGE FOUND ON PAGE {page_number}
IMAGE INDEX: {image["image_index"]}
IMAGE PATH: {image["image_path"]}

IMAGE OCR TEXT:
{image["image_ocr_text"]}
""".strip()

    doc = Document(
        page_content=image_text,
        metadata={
            "source": PDF_PATH,
            "page_number": page_number,
            "content_type": "image",
            "image_index": image["image_index"],
            "image_path": image["image_path"],
            "image_ext": image["image_ext"],
        }
    )

    langchain_docs.append(doc)


print("Total LangChain Documents created:", len(langchain_docs))


Total LangChain Documents created: 53


In [20]:
# ============================================================
# 6. Save Parsed Output
# ============================================================

# Save page records
with open(OUTPUT_DIR / "page_records.json", "w", encoding="utf-8") as f:
    json.dump(page_records, f, indent=2, ensure_ascii=False)

# Save image records
with open(OUTPUT_DIR / "image_records.json", "w", encoding="utf-8") as f:
    json.dump(image_records, f, indent=2, ensure_ascii=False)

# Save table records
with open(OUTPUT_DIR / "table_records.json", "w", encoding="utf-8") as f:
    json.dump(table_records, f, indent=2, ensure_ascii=False)

# Save all LangChain document content as markdown
with open(OUTPUT_DIR / "rag_ready_documents.md", "w", encoding="utf-8") as f:
    for i, doc in enumerate(langchain_docs):
        f.write(f"\n\n# Document {i + 1}\n")
        f.write(f"\nMetadata:\n```json\n{json.dumps(doc.metadata, indent=2)}\n```\n")
        f.write("\nContent:\n")
        f.write(doc.page_content)
        f.write("\n\n---\n")

# Save table markdown separately
with open(OUTPUT_DIR / "extracted_tables.md", "w", encoding="utf-8") as f:
    for table in table_records:
        f.write(f"\n\n## Page {table['page_number']} - Table {table['table_index']}\n\n")
        f.write(table["markdown"])
        f.write("\n\n---\n")

print("\nSaved outputs in:", OUTPUT_DIR)



Saved outputs in: parsed_complex_pdf_output


In [21]:
# ============================================================
# 7. Preview Parsed Output
# ============================================================

print("\n================ PAGE TEXT PREVIEW ================\n")
print(langchain_docs[0].page_content[:1500])

print("\n================ TABLE PREVIEW ================\n")
table_docs = [doc for doc in langchain_docs if doc.metadata["content_type"] == "table"]

if table_docs:
    print(table_docs[0].page_content[:1500])
else:
    print("No table docs found.")

print("\n================ IMAGE OCR PREVIEW ================\n")
image_docs = [doc for doc in langchain_docs if doc.metadata["content_type"] == "image"]

if image_docs:
    print(image_docs[0].page_content[:1500])
else:
    print("No image docs found.")



================ PAGE TEXT PREVIEW ================

PAGE 1

SELECTABLE TEXT:
Complex RAG Parsing Sample - synthetic document
Page 1
Complex Document for RAG Parsing Tests
Synthetic 15-page PDF with paragraphs, simple and complex tables, diagrams, scanned-form style image, metadata
examples, and production RAG edge cases.
Story Line
Three client teams - Arka Finance, BlueLeaf Retail, and CityRide Mobility - are migrating contracts, policies, support records,
and operational reports into a single RAG platform. Each team has different document types, access rules, and parsing
challenges. The RAG system must answer questions with citations while ensuring that one client never sees another clients
data.
This PDF is intentionally designed to test document loaders, PDF parsers, OCR workflows, table extraction, chunking
strategies, metadata preservation, and source citation quality.
Key statement: RAG does not train the model. RAG gives the model the right context before answering.
Section
P

In [23]:
!pip install -q docling langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 995.7/995.7 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.8/317.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [29]:
# if you run into pillow ImportError: cannot import name '_Ink' from 'PIL._typing'
# !pip uninstall -y pillow docling docling-core
# !pip install --no-cache-dir pillow docling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 147.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.8/317.8 kB 225.6 MB/s eta 0:00:00


In [1]:
# # ============================================================
# # Complex PDF Parsing using Docling
# # PDF -> DoclingDocument -> Markdown / JSON / LangChain Documents
# # ============================================================

# # Install dependencies
# !pip install -q docling langchain-core langchain-text-splitters pandas

import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
# ============================================================
# 1. Input PDF Path
# ============================================================

PDF_PATH = "complex_rag_parsing_sample_with_sunny_image.pdf"

OUTPUT_DIR = Path("docling_parsed_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PDF path:", PDF_PATH)

PDF path: complex_rag_parsing_sample_with_sunny_image.pdf


In [3]:
! pip show pillow

Name: pillow
Version: 12.3.0
Summary: Python Imaging Library (fork)
Home-page: https://python-pillow.github.io
Author: 
Author-email: Jeffrey 'Alex' Clark <aclark@aclark.net>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: bokeh, diffusers, docling-core, docling-ibm-models, docling-parse, dopamine_rl, fastai, gradio, ImageIO, matplotlib, pdfplumber, pytesseract, python-pptx, rapidocr, scikit-image, tensorboard, torchtune, torchvision, wordcloud


In [4]:
# ============================================================
# 2. Parse PDF using Docling
# ============================================================

from docling.document_converter import DocumentConverter

converter = DocumentConverter()

# Convert PDF into Docling structured document
conversion_result = converter.convert(PDF_PATH)

# Main Docling document object
docling_doc = conversion_result.document

print("Docling conversion completed.")


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[INFO] 2026-10-14 06:17:39,180 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-10-14 06:17:39,196 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-10-14 06:17:39,200 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.10.0/torch/PP-OCRv6/det/PP-OCRv6_det_small.pth
[INFO] 2026-10-14 06:17:41,011 [RapidOCR] download_file.py:82: Download size: 9.77MB
[INFO] 2026-10-14 06:17:41,474 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-10-14 06:17:41,486 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Conv

Docling conversion completed.


In [5]:
# ============================================================
# 3. Export Parsed Output to Markdown
# ============================================================

markdown_text = docling_doc.export_to_markdown()

markdown_path = OUTPUT_DIR / "parsed_docling_output.md"

with open(markdown_path, "w", encoding="utf-8") as f:
    f.write(markdown_text)

print("Markdown saved at:", markdown_path)


Markdown saved at: docling_parsed_output/parsed_docling_output.md


In [6]:
# ============================================================
# 4. Export Parsed Output to JSON / Dict
# ============================================================

# Docling versions may expose export_to_dict() or model_dump()
# This fallback handles both styles.

try:
    doc_dict = docling_doc.export_to_dict()
except Exception:
    try:
        doc_dict = docling_doc.model_dump()
    except Exception:
        doc_dict = {"raw_string": str(docling_doc)}

json_path = OUTPUT_DIR / "parsed_docling_output.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(doc_dict, f, indent=2, ensure_ascii=False)

print("JSON saved at:", json_path)

JSON saved at: docling_parsed_output/parsed_docling_output.json


In [7]:
# ============================================================
# 5. Create LangChain Document from Full Markdown
# ============================================================

full_doc = Document(
    page_content=markdown_text,
    metadata={
        "source": PDF_PATH,
        "parser": "docling",
        "content_type": "full_document_markdown"
    }
)

print("LangChain full document created.")
print("Characters:", len(full_doc.page_content))

LangChain full document created.
Characters: 33796


In [ ]:
# # ============================================================
# # 6. Split Markdown into RAG-ready Chunks
# # ============================================================

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1200,
#     chunk_overlap=200,
#     separators=[
#         "\n# ",
#         "\n## ",
#         "\n### ",
#         "\n\n",
#         "\n",
#         " ",
#         ""
#     ]
# )

# chunks = text_splitter.split_documents([full_doc])

# print("Total chunks created:", len(chunks))

# for i, chunk in enumerate(chunks[:3]):
#     print(f"\n--- Chunk {i + 1} ---")
#     print("Metadata:", chunk.metadata)
#     print(chunk.page_content[:800])


# # ============================================================
# # 7. Save RAG-ready Chunks as JSONL
# # ============================================================

# chunks_jsonl_path = OUTPUT_DIR / "rag_ready_chunks.jsonl"

# with open(chunks_jsonl_path, "w", encoding="utf-8") as f:
#     for i, chunk in enumerate(chunks):
#         record = {
#             "chunk_id": i + 1,
#             "page_content": chunk.page_content,
#             "metadata": chunk.metadata
#         }
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")

# print("Chunks JSONL saved at:", chunks_jsonl_path)


# # ============================================================
# # 8. Save RAG-ready Chunks as Markdown Preview
# # ============================================================

# chunks_md_path = OUTPUT_DIR / "rag_ready_chunks_preview.md"

# with open(chunks_md_path, "w", encoding="utf-8") as f:
#     for i, chunk in enumerate(chunks):
#         f.write(f"\n\n# Chunk {i + 1}\n")
#         f.write("```json\n")
#         f.write(json.dumps(chunk.metadata, indent=2, ensure_ascii=False))
#         f.write("\n```\n\n")
#         f.write(chunk.page_content)
#         f.write("\n\n---\n")

# print("Chunks markdown preview saved at:", chunks_md_path)


# # ============================================================
# # 9. Basic Search Test on Parsed Markdown
# # ============================================================

# keywords = [
#     "Appendix A",
#     "Complex Clause Responsibility Matrix",
#     "Appendix G",
#     "Scanned Tax Invoice",
#     "Appendix H",
#     "Scanned Utility Bill",
#     "Appendix I",
#     "Profile Image",
#     "Retention Heatmap",
#     "Vector Database"
# ]

# print("\nKeyword check:")
# for keyword in keywords:
#     found = keyword.lower() in markdown_text.lower()
#     print(f"{keyword}: {found}")


# # ============================================================
# # 10. Final Output Summary
# # ============================================================

# print("\nDONE.")
# print("Output folder:", OUTPUT_DIR)
# print("Files created:")
# print("-", markdown_path.name)
# print("-", json_path.name)
# print("-", chunks_jsonl_path.name)
# print("-", chunks_md_path.name)

# print("\nUse 'chunks' for embeddings/vector database.")

In [ ]:
# !pip install -q langchain-docling langchain-text-splitters

# from langchain_docling import DoclingLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF_PATH = "/mnt/data/complex_rag_parsing_sample_with_sunny_image.pdf"

# loader = DoclingLoader(file_path=PDF_PATH)

# docs = loader.load()

# print("Total docs:", len(docs))
# print(docs[0].page_content[:1000])
# print(docs[0].metadata)

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1200,
#     chunk_overlap=200
# )

# chunks = splitter.split_documents(docs)

# print("Total chunks:", len(chunks))